# k-mamba — Entraînement sur Kaggle GPU

Ce notebook entraîne un modèle Mamba 1.45M paramètres sur des données textuelles françaises.

**Configuration** : GPU Tesla T4 / P100 (16GB VRAM)

## 1. Configuration GPU

In [ ]:
# Vérifier GPU disponible
!nvidia-smi
!nvcc --version

## 2. Installation des dépendances

In [ ]:
# Mettre à jour et installer dépendances
!apt-get update -qq
!apt-get install -y -qq gcc g++ nasm git make

# Vérifier installations
!gcc --version
!nasm --version
!make --version

## 3. Clone k-mamba et compilation

In [ ]:
# Clone le repo (adapter l'URL selon ton repo)
%cd /kaggle/working
!rm -rf k-mamba
!git clone https://github.com/goldensam777/k-mamba.git
%cd k-mamba

In [ ]:
# Compilation avec CUDA
!make clean
!make lib 2>&1 | tail -20
!make models/kmamba_cuda 2>&1 | tail -10

## 4. Préparation des données

In [ ]:
# Option A: Télécharger un corpus français (ex: Le Monde, Wikipedia, etc.)
# Option B: Uploader ton propre fichier via l'interface Kaggle à droite →

# Exemple avec un dataset public:
!mkdir -p data
!wget -q -O data/corpus_fr.txt "URL_DU_CORPUS" 2>/dev/null || echo "À remplacer par ton corpus"

# Vérifier taille
!ls -lh data/

## 5. Entraînement

In [ ]:
# Configuration d'entraînement
CONFIG = {
    'vocab_size': 32768,     # BPE 32K
    'dim': 128,              # 128 dimensions
    'state_size': 256,       # 256 états SSM
    'n_layers': 4,           # 4 blocs Mamba
    'seq_len': 512,          # 512 tokens par séquence
    'batch_size': 16,        # 16 séquences par batch
    'lr': 2e-4,              # Learning rate MUON
    'total_tokens': 10_000_000,  # 10M tokens (Chinchilla)
    'data_file': 'data/corpus_fr.txt',
    'checkpoint': 'kmamba_kaggle.bin',
    'log_dir': 'logs'
}

print(f"Modèle: {1.45:.2f}M paramètres")
print(f"Tokens cible: {CONFIG['total_tokens']:,}")
print(f"Batch: {CONFIG['batch_size']} × {CONFIG['seq_len']} = {CONFIG['batch_size'] * CONFIG['seq_len']:,} tokens/step")

In [ ]:
# Lancer l'entraînement GPU
!mkdir -p {CONFIG['log_dir']}

!./models/kmamba_cuda train {CONFIG['data_file']} {CONFIG['checkpoint']} {CONFIG['log_dir']} \
    --total-tokens {CONFIG['total_tokens']} 2>&1 | tee {CONFIG['log_dir']}/train.log

## 6. Visualisation des résultats

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Charger les logs d'entraînement
try:
    steps = pd.read_csv(f"{CONFIG['log_dir']}/step.csv")
    epochs = pd.read_csv(f"{CONFIG['log_dir']}/epoch.csv")
    
    # Plot loss
    plt.figure(figsize=(12, 5))
    
    plt.subplot(1, 2, 1)
    plt.plot(steps['step'], steps['loss'])
    plt.xlabel('Step')
    plt.ylabel('Loss')
    plt.title('Loss par step')
    plt.grid(True)
    
    plt.subplot(1, 2, 2)
    plt.plot(epochs['epoch'], epochs['train_bt'], label='Train')
    plt.plot(epochs['epoch'], epochs['val'], label='Val')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Loss par epoch')
    plt.legend()
    plt.grid(True)
    
    plt.tight_layout()
    plt.show()
except Exception as e:
    print(f"Logs non disponibles: {e}")

## 7. Sauvegarde du checkpoint

In [ ]:
# Vérifier checkpoint
!ls -lh {CONFIG['checkpoint']}

# Sauvegarder vers /kaggle/output pour téléchargement
!cp {CONFIG['checkpoint']} /kaggle/output/
!cp -r {CONFIG['log_dir']} /kaggle/output/
print("Checkpoint sauvegardé dans /kaggle/output/")